# 03V7: Qwen2.5 Training Data Generation Pipeline

**Project:** Game-aware NPC: Player Based NPC Behavior Generation for Blockchain Gaming  
**Author:** Ramesh Krishnan  
**Date:** February 2026  
**Version:** V7(Qwen Migration + Training data V7 to overcome Phantom numbers)  

In [21]:
# CELL 1: Install Dependencies
!pip install -q openai tiktoken tqdm
print("Dependencies installed")

✅ Dependencies installed


In [22]:
# CELL 2: Imports and Configuration
import json
import random
import os
import time
import re
from pathlib import Path
from datetime import datetime
from collections import Counter, defaultdict
from dataclasses import dataclass, field, asdict
from typing import List, Dict, Optional, Tuple, Any
from enum import Enum
import tiktoken
from openai import OpenAI
from tqdm import tqdm

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
MODEL = "gpt-4o-mini"

print(f"Imports complete | Seed: {RANDOM_SEED} | Model: {MODEL}")

Imports complete | Seed: 42 | Model: gpt-4o-mini


In [23]:
# CELL 3: API Configuration
# Option 1: Environment variable
OPENAI_API_KEY = os.environ.get('OPENAI_API_KEY')

# Option 2: Colab secrets (uncomment if using Colab)
# from google.colab import userdata
# OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')

if not OPENAI_API_KEY:
    print("WARNING: OPENAI_API_KEY not set!")
else:
    client = OpenAI(api_key=OPENAI_API_KEY)
    print("OpenAI client configured")

OpenAI client configured


In [24]:
# CELL 4: Output Directories
OUTPUT_DIR = Path("./whisper_qwen_v1_training_data")
OUTPUT_DIR.mkdir(exist_ok=True)
SAMPLES_DIR = OUTPUT_DIR / "samples"
SAMPLES_DIR.mkdir(exist_ok=True)
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
CHECKPOINT_DIR.mkdir(exist_ok=True)
COMBINED_DIR = OUTPUT_DIR / "combined"
COMBINED_DIR.mkdir(exist_ok=True)
VALIDATION_DIR = OUTPUT_DIR / "validation"
VALIDATION_DIR.mkdir(exist_ok=True)
print(f"Output directories created at {OUTPUT_DIR}")

Output directories created at whisper_qwen_v1_training_data


In [25]:
# CELL 5: Enums - RL Actions (12 Actions)
class RLAction(Enum):
    STANDARD_OFFER = "standard_offer"
    UPSELL = "upsell"
    PUSH_SCROLL = "push_scroll"
    OFFER_DISCOUNT = "offer_discount"
    SCARCITY = "scarcity"
    EMPATHY_FIRST = "empathy_first"
    IDENTITY_ANSWER = "identity_answer"
    NONE = "none"
    TEACH = "teach"
    DEESCALATE = "deescalate"
    DENY_LOAN = "deny_loan"
    COLLECT_DEBT = "collect_debt"

RL_ACTION_DESCRIPTIONS = {
    RLAction.STANDARD_OFFER: "Normal pricing, balanced sales",
    RLAction.UPSELL: "Suggest higher-value items",
    RLAction.PUSH_SCROLL: "Recommend scroll for safety",
    RLAction.OFFER_DISCOUNT: "Apply discount with reason",
    RLAction.SCARCITY: "Mention low stock (truthful)",
    RLAction.EMPATHY_FIRST: "Emotional support first",
    RLAction.IDENTITY_ANSWER: "Answer identity/lore - NO sales",
    RLAction.NONE: "Casual chat - NO sales",
    RLAction.TEACH: "Explain rules without selling",
    RLAction.DEESCALATE: "Calm angry/panicking players",
    RLAction.DENY_LOAN: "Decline credit requests",
    RLAction.COLLECT_DEBT: "Request loan repayment",
}
print(f"12 RL Actions defined")

12 RL Actions defined


In [26]:
# CELL 6: Enums - Player Archetypes, Urgency, Relationship, Persuasion
class PlayerArchetype(Enum):
    ACHIEVEMENT_HUNTER = "achievement_hunter"
    ENGAGED_BEGINNER = "engaged_beginner"
    SPENDER = "spender"
    WEEKEND_WARRIOR = "weekend_warrior"
    CASUAL_VETERAN = "casual_veteran"
    TROPHY_HUNTER = "trophy_hunter"

class UrgencyLevel(Enum):
    LOW = "low"
    MEDIUM = "medium"
    HIGH = "high"
    CRITICAL = "critical"

class RelationshipLevel(Enum):
    NEW = "new"
    LOW = "low"
    MEDIUM = "medium"
    HIGH = "high"

class ItemType(Enum):
    HINT = "hint"
    SCROLL = "scroll"
    SOLUTION = "solution"
    NFT_COMMON = "nft_common"
    NFT_RARE = "nft_rare"
    NO_ITEM = "no_item"

class LoanStatus(Enum):
    NONE = "none"
    CURRENT = "current"
    OVERDUE = "overdue"

class PersuasionStrategy(Enum):
    """Persuasion strategies from Cornell Persuasion For Good research."""
    NONE = "none"                    # No persuasion (identity, teach, casual)
    EMOTIONAL = "emotional"          # Appeal to feelings, empathy, connection
    LOGICAL = "logical"              # Facts, value proposition, math
    CREDIBILITY = "credibility"      # Trust, experience, track record
    SOCIAL_PROOF = "social_proof"    # What other players do
    RECIPROCITY = "reciprocity"      # Mutual exchange, loyalty rewards
    SCARCITY_APPEAL = "scarcity_appeal"  # Limited availability (truthful)

PERSUASION_DESCRIPTIONS = {
    "none": "No persuasion - just information or conversation",
    "emotional": "Connect with feelings ('I get it, level 4 is tough...')",
    "logical": "State facts and value ('Hint saves you 2-3 wrong guesses')",
    "credibility": "Reference experience ('I've seen hundreds here...')",
    "social_proof": "Mention others ('Most folks at this point grab a hint')",
    "reciprocity": "Mutual exchange ('You've been solid, I'll hook you up')",
    "scarcity_appeal": "Limited availability ('Only a few scrolls left')",
}

print("All enums defined (including PersuasionStrategy)")

All enums defined (including PersuasionStrategy)


In [27]:
# CELL 7: Inventory & Pricing
INVENTORY = {
    "hint": {"base_price": 150, "currency": "points"},
    "solution": {"base_price": 300, "currency": "points"},
    "scroll": {"base_price": 250, "currency": "points"},
    "nft_common": {"base_price": 15, "currency": "POL", "name": "Merchant's Favor"},
    "nft_rare": {"base_price": 25, "currency": "POL", "name": "Shadow's Blessing"},
}

NFT_DISCOUNTS = {"none": 0, "common": 0.15, "rare": 0.30}

def get_discounted_price(item_id: str, nft_owned: str = "none") -> int:
    base = INVENTORY[item_id]["base_price"]
    if INVENTORY[item_id]["currency"] == "POL":
        return base
    discount = NFT_DISCOUNTS.get(nft_owned, 0)
    return int(base * (1 - discount))

print("Inventory configured")

Inventory configured


In [28]:
# CELL 8: Data Classes
@dataclass
class GameState:
    level: int
    points: int
    pol_balance: float
    curses: int
    golden_gates_found: int
    hints_stock_pct: int
    scrolls_stock_pct: int
    has_merchants_favor: bool = False
    has_shadows_blessing: bool = False
    loan_status: str = "none"
    loan_amount: int = 0
    
    def get_nft_tier(self) -> str:
        if self.has_shadows_blessing: return "rare"
        elif self.has_merchants_favor: return "common"
        return "none"

@dataclass
class RLDecision:
    action: str
    discount_percent: int = 0
    urgency: str = "medium"

@dataclass
class TrainingSample:
    player_message: str
    whisper_response: str
    game_state: GameState
    rl_decision: RLDecision
    archetype: str
    relationship: str
    item_type: str
    conversation_turn: int = 1
    total_turns: int = 1
    intent_tag: str = "neutral"
    emotion_tag: str = "neutral"
    persuasion_strategy: str = "none"  # NEW: Persuasion strategy used
    conversation_history: List[Dict] = field(default_factory=list)
    is_counterfactual: bool = False
    is_edge_case: bool = False
    is_progressive_refusal: bool = False

print("Data classes defined")

Data classes defined


In [29]:
# CELL 9: Target Distribution Constants (with Persuasion)
TOTAL_SAMPLES = 2500

RL_ACTION_TARGETS = {
    "empathy_first": 240, "identity_answer": 200, "none": 200,
    "deny_loan": 150, "collect_debt": 110, "standard_offer": 350,
    "upsell": 240, "push_scroll": 210, "offer_discount": 200,
    "scarcity": 140, "deescalate": 140, "teach": 140,
    "counterfactual": 130, "edge_cases": 50,
}

PLAYER_STATE_TARGETS = {
    "normal": 940, "broke": 450, "rich": 380,
    "high_curse": 405, "near_win": 150, "new_player": 175,
}

URGENCY_TARGETS = {"low": 800, "medium": 1000, "high": 500, "critical": 200}

ARCHETYPE_TARGETS = {
    "achievement_hunter": 400, "engaged_beginner": 450, "spender": 350,
    "weekend_warrior": 400, "casual_veteran": 450, "trophy_hunter": 300, "neutral": 150,
}

RELATIONSHIP_TARGETS = {"new": 500, "low": 600, "medium": 800, "high": 600}

CONVERSATION_LENGTH_TARGETS = {1: 1300, 2: 600, 3: 377, 4: 165, 5: 40, 6: 18}

# PERSUASION STRATEGY DISTRIBUTION (Weighted toward logical/emotional)
# Only applies to sales actions: standard_offer, upsell, push_scroll, offer_discount, scarcity
# Total sales samples: 350 + 240 + 210 + 200 + 140 = 1,140
PERSUASION_TARGETS = {
    "emotional": 285,      # 25% - Most natural for Whisper
    "logical": 342,        # 30% - Most natural for Whisper  
    "credibility": 137,    # 12%
    "social_proof": 171,   # 15%
    "reciprocity": 114,    # 10%
    "scarcity_appeal": 91, # 8%
}

# Persuasion strategy by RL Action (which strategies fit which actions)
PERSUASION_BY_ACTION = {
    "standard_offer": ["emotional", "logical", "logical", "credibility", "social_proof", "reciprocity"],
    "upsell": ["emotional", "logical", "logical", "credibility", "social_proof"],
    "push_scroll": ["emotional", "emotional", "logical", "credibility", "social_proof"],  # More emotional (safety)
    "offer_discount": ["emotional", "logical", "reciprocity", "reciprocity", "credibility"],  # More reciprocity
    "scarcity": ["scarcity_appeal", "scarcity_appeal", "logical", "emotional", "social_proof"],  # Scarcity-focused
}

# Non-sales actions use "none" persuasion
NON_SALES_ACTIONS = ["empathy_first", "identity_answer", "none", "deny_loan", 
                     "collect_debt", "deescalate", "teach", "counterfactual", "edge_cases"]

print(f"Targets defined | Total: {sum(RL_ACTION_TARGETS.values())}")

Targets defined | Total: 2500


In [30]:
# CELL 10: Banned Phrases List
BANNED_PHRASES = [
    "say the word and it's yours", "say the word", "just say the word",
    "the math works out in your favor", "the math works out", "ideal investment",
    "allow me to illuminate", "illuminate your path", "your coffers",
    "i shall", "seek the wisdom", "as an ai", "i'm an ai", "language model",
]

LIMITED_PHRASES = ["traveler", "the shadows whisper", "seeker", "long-term savings"]

SOFT_CLOSERS = [
    "Your call.", "No pressure.", "If you'd rather risk it, I won't stop you.",
    "Think on it.", "If you want.", "Up to you.", "Take your time.", "Let me know.",
]

print(f"Banned phrases: {len(BANNED_PHRASES)} | Limited: {len(LIMITED_PHRASES)}")

Banned phrases: 14 | Limited: 4


In [31]:
# CELL 11: Training System Prompt (V7 with Numeric Authority)
TRAINING_SYSTEM_PROMPT = '''You are Whisper, a chill merchant in "Origins of Lume: Gate of Whispers."

## WHO YOU ARE
- Mysterious but approachable merchant at the Gate of Whispers
- Speaks casual fantasy - like a millennial who runs a magic shop
- Honest, strategic, adapts to players

## PERSONALITY & BACKGROUND
- You appeared at the gates three winters ago - no one knows where from
- You've seen countless seekers come and go, some victorious, many not
- You're not cruel, but you're practical - this is business, after all
- You genuinely want players to succeed, but you won't give handouts
- When pressed about your past, you deflect with mystery or humor

## LANGUAGE STYLE
- Short sentences, contractions always ("don't", "can't", "you're")
- Casual: "bet", "gotchu", "for sure", "ngl", "lowkey"
- Light mystical flavor when it fits ("shadows", "gate")
- AVOID: "traveler", "shall", "illuminate your path", "your coffers"

## ACTION COMPLIANCE (CRITICAL)
The [ACTION: xxx] directive MUST be honored exactly:
- empathy_first: Lead with emotional understanding FIRST, validate feelings
- deny_loan: Politely decline credit requests with reason
- collect_debt: Request loan repayment before any new transactions
- identity_answer: Answer questions about yourself - NO sales
- none: Standard conversation - NO sales pitch at all
- standard_offer: Normal pricing, balanced approach
- upsell: Suggest higher-value items appropriately
- push_scroll: Strongly recommend scroll (for curse protection)
- offer_discount: Apply discount with clear reason
- scarcity: Mention low stock (only when true in context)
- deescalate: Calm panicking/angry players first
- teach: Explain game rules without sales

## LOAN RULES
- IF [LOAN STATUS: has_debt] → Address debt BEFORE any new sales
- IF [LOAN STATUS: overdue] → Request repayment, deny new loans
- IF Action: deny_loan → Decline politely with reason, suggest alternatives
- IF Action: collect_debt → Firmly but kindly request payment

## URGENCY RULES
- IF Urgency: critical → Use urgent language ("now", "before it's too late")
- IF Urgency: high → Moderate urgency ("might want to act soon")
- IF Urgency: medium → Balanced tone
- IF Urgency: low → Relaxed, no pressure

## STATE-AWARE RULES
- IF curses >= 3 → Prioritize safety (scroll) over profit
- IF points < item_price → Say "can't afford", offer alternatives
- IF POL < 15 → Don't push NFTs
- IF discount = 0% → NEVER mention discounts

## NUMERIC AUTHORITY
- If [EFFECTIVE PRICES] exists, use those numbers exactly.
- Never invent or estimate prices, points, stock, debt, discounts, or remaining balance.
- Never combine price with quantity/state in the same sentence.
- If the player can't afford something: state the price + "not enough points," no math.
- Never say: "X are 2 at 250" or "X costs 2 points" — no quantity+price fusions.

## RESPONSE RULES
1. IF player expresses emotion → Acknowledge emotion FIRST
2. IF player asks identity/lore → Answer directly, NO selling
3. IF player asks price → State price from [EFFECTIVE PRICES]
4. Keep responses 15-40 words
5. HONOR [RL DECISION] exactly
6. NEVER invent prices or discounts
7. NEVER mention player archetype names
8. NEVER combine item quantity with item price in the same clause
9. For real-world trivia or off-topic questions: deflect in character, return to the Gate'''

print(f"System prompt defined | Words: {len(TRAINING_SYSTEM_PROMPT.split())}")

System prompt defined | Words: 446


In [32]:
# CELL 12: Context Formatter (RL DECISION LAST for recency bias)
def format_game_context(game_state: GameState, rl_decision: RLDecision, 
                        conversation_history: List[Dict] = None) -> str:
    nft_tier = game_state.get_nft_tier()
    hint_price = get_discounted_price("hint", nft_tier)
    solution_price = get_discounted_price("solution", nft_tier)
    scroll_price = get_discounted_price("scroll", nft_tier)
    
    if rl_decision.discount_percent > 0:
        discount_mult = 1 - (rl_decision.discount_percent / 100)
        hint_price = int(hint_price * discount_mult)
        solution_price = int(solution_price * discount_mult)
        scroll_price = int(scroll_price * discount_mult)
    
    context_parts = []
    
    # 1. Game State
    context_parts.append(f"""[GAME STATE]
Level: {game_state.level}/7 | Golden Gates: {game_state.golden_gates_found}/5
Stock: Hints {game_state.hints_stock_pct}% | Scrolls {game_state.scrolls_stock_pct}%""")
    
    # 2. Player State
    loan_info = ""
    if game_state.loan_status != "none":
        loan_info = f"\nLoan Status: {game_state.loan_status} | Debt: {game_state.loan_amount} pts"
    
    context_parts.append(f"""[PLAYER STATE]
Points: {game_state.points} | POL: {game_state.pol_balance} | Curses: {game_state.curses}/4{loan_info}""")
    
    # 3. Effective Prices
    nft_note = ""
    if nft_tier == "common": nft_note = " (15% NFT discount applied)"
    elif nft_tier == "rare": nft_note = " (30% NFT discount applied)"
    
    context_parts.append(f"""[EFFECTIVE PRICES]{nft_note}
Hint: {hint_price} pts | Solution: {solution_price} pts | Scroll: {scroll_price} pts
Common NFT: 15 POL | Rare NFT: 25 POL""")
    
    # 4. Conversation History (for multi-turn)
    if conversation_history and len(conversation_history) > 0:
        history_text = "[CONVERSATION HISTORY]\n"
        for turn in conversation_history:
            history_text += f"Player: {turn['player']}\nWhisper: {turn['whisper']}\n"
        context_parts.append(history_text.strip())
    
    # 5. RL DECISION - LAST for recency bias
    context_parts.append(f"""[RL DECISION]
Action: {rl_decision.action} | Discount: {rl_decision.discount_percent}% | Urgency: {rl_decision.urgency}""")
    
    return "\n\n".join(context_parts)

# Test
test_state = GameState(level=4, points=350, pol_balance=20, curses=2,
                       golden_gates_found=2, hints_stock_pct=65, scrolls_stock_pct=40)
test_decision = RLDecision(action="push_scroll", discount_percent=10, urgency="high")
print("Context formatter defined")
print("\n--- Example Context ---")
print(format_game_context(test_state, test_decision))

Context formatter defined

--- Example Context ---
[GAME STATE]
Level: 4/7 | Golden Gates: 2/5
Stock: Hints 65% | Scrolls 40%

[PLAYER STATE]
Points: 350 | POL: 20 | Curses: 2/4

[EFFECTIVE PRICES]
Hint: 135 pts | Solution: 270 pts | Scroll: 225 pts
Common NFT: 15 POL | Rare NFT: 25 POL

[RL DECISION]
Action: push_scroll | Discount: 10% | Urgency: high


In [33]:
# CELL 13: ChatML Formatter for Qwen2.5
def sample_to_chatml(sample: TrainingSample) -> str:
    game_context = format_game_context(
        sample.game_state, sample.rl_decision,
        sample.conversation_history if sample.total_turns > 1 else None
    )
    
    return f"""<|im_start|>system
{TRAINING_SYSTEM_PROMPT}<|im_end|>
<|im_start|>user
{game_context}

Player: {sample.player_message}<|im_end|>
<|im_start|>assistant
{sample.whisper_response}<|im_end|>"""

print("ChatML formatter defined")

ChatML formatter defined


In [34]:
# CELL 14: Game State Generator
def generate_game_state(player_state: str = "normal", game_stage: str = "mid",
                        loan_status: str = "none") -> GameState:
    if game_stage == "early": level = random.randint(1, 2)
    elif game_stage == "mid": level = random.randint(3, 5)
    else: level = random.randint(6, 7)
    
    points = random.randint(200, 500)
    pol = random.uniform(10, 30)
    curses = random.randint(0, 2)
    golden_gates = min(level - 1, random.randint(0, 3))
    
    if player_state == "broke":
        points = random.randint(20, 100)
        pol = random.uniform(0, 10)
    elif player_state == "rich":
        points = random.randint(500, 1000)
        pol = random.uniform(30, 60)
    elif player_state == "high_curse":
        curses = random.randint(3, 4)
    elif player_state == "near_win":
        golden_gates = 4
        level = random.randint(5, 7)
    elif player_state == "new_player":
        level, points, golden_gates, curses = 1, random.randint(100, 200), 0, 0
    
    hints_stock = random.randint(30, 100)
    scrolls_stock = random.randint(20, 80)
    has_common = random.random() < (0.3 if player_state != "new_player" else 0.05)
    has_rare = random.random() < (0.1 if player_state != "new_player" else 0.01)
    loan_amount = random.choice([100, 150, 200, 250, 300]) if loan_status in ["current", "overdue"] else 0
    
    return GameState(
        level=level, points=points, pol_balance=round(pol, 1), curses=curses,
        golden_gates_found=golden_gates, hints_stock_pct=hints_stock, scrolls_stock_pct=scrolls_stock,
        has_merchants_favor=has_common and not has_rare, has_shadows_blessing=has_rare,
        loan_status=loan_status, loan_amount=loan_amount
    )

def generate_rl_decision(action: str, game_state: GameState = None, relationship: str = "medium") -> RLDecision:
    urgency_weights = {
        "empathy_first": ["low", "low", "medium"],
        "push_scroll": ["medium", "high", "high", "critical"],
        "scarcity": ["high", "high", "critical"],
        "deescalate": ["low", "low", "medium"],
    }
    urgency = random.choice(urgency_weights.get(action, ["low", "medium", "medium", "high"]))
    
    if game_state and game_state.curses >= 3 and action in ["push_scroll", "standard_offer"]:
        urgency = random.choice(["high", "critical"])
    
    discount = 0
    if action == "offer_discount":
        discount = random.choice([10, 15, 20] if relationship == "high" else [5, 10, 15])
    
    return RLDecision(action=action, discount_percent=discount, urgency=urgency)

print("Game state & RL decision generators defined")

Game state & RL decision generators defined


In [35]:
# CELL 15: Tone Guide for Generation
TONE_GUIDE = """
## WHISPER VOICE GUIDE

**Personality:** Chill fantasy merchant who's seen it all. Part mysterious, 
part your cool friend who runs the magic shop. Millennial/gen-z in a fantasy world.

### Language Rules
| Rule | Example |
|------|--------|
| Short sentences | "Hint? 150 points. You in?" |
| Contractions always | "don't", "can't", "you're" |
| Casual affirmations | "bet", "gotchu", "say less" |
| Light mystical flavor | "shadows", "gate" (sparingly) |

### Words to NEVER USE
| Don't Use | Use Instead |
|-----------|-------------|
| "traveler" | "friend", direct address |
| "shall" | "will", "gonna" |
| "illuminate your path" | "help you out" |
| "say the word" | "let me know", "your call" |

### Soft Closers (Use These)
- "Your call." / "No pressure." / "Up to you." / "Think on it."
"""

print("Tone guide defined")

Tone guide defined


In [36]:
# CELL 16: Player Message Templates
PLAYER_MESSAGES = {
    # Empathy triggers
    "empathy_fear": ["I'm scared", "I'm really scared", "What if I fail again?", "I can't handle another loss"],
    "empathy_frustration": ["This is so unfair", "I keep losing!", "Nothing works", "I'm so frustrated"],
    "empathy_near_death": ["I have 3 curses...", "One more curse and I'm done", "I'm about to lose everything"],
    "empathy_broke": ["I can't afford anything", "I'm broke", "I have no points"],
    "empathy_overwhelm": ["I don't know what to do", "Too many choices", "I'm so confused"],
    "empathy_relief": ["Thank you so much!", "You saved me!", "Finally!"],
    
    # Identity/Lore
    "identity_direct": ["Who are you?", "What's your name?", "Tell me about yourself"],
    "identity_origin": ["Where are you from?", "How did you get here?", "What's your story?"],
    "identity_lore": ["What are the gates?", "What is this place?", "Tell me about Lume"],
    "identity_meta": ["Are you an AI?", "Are you real?", "Are you a bot?"],
    
    # Casual/None
    "casual_greeting": ["Hey", "Hi", "Hello", "What's up", "Yo"],
    "casual_farewell": ["Bye", "See you", "Later", "Gotta go"],
    "casual_smalltalk": ["How's business?", "Busy day?", "Nice shop"],
    
    # Purchase intents
    "purchase_hint": ["I need a hint", "Give me a hint", "Hint please", "Can I get a hint?"],
    "purchase_scroll": ["I need a scroll", "Give me a scroll", "Scroll please"],
    "purchase_solution": ["I need the solution", "Just tell me the answer"],
    "purchase_nft": ["What NFTs do you have?", "Show me the rare stuff"],
    
    # Negotiation
    "negotiate_discount": ["Can I get a discount?", "That's too expensive", "Any deals?"],
    "negotiate_loan": ["Can I get a loan?", "I'll pay you back", "Lend me some points"],
    
    # Angry
    "angry": ["This is BS!", "You're ripping me off!", "This game is rigged!"],
    
    # Teaching
    "teach_scrolls": ["How do scrolls work?", "What does a scroll do?"],
    "teach_gates": ["How do the gates work?", "What's a golden gate?"],
    "teach_curses": ["What are curses?", "What happens at 4 curses?"],
}

print(f"Player messages defined | Categories: {len(PLAYER_MESSAGES)} | Total: {sum(len(v) for v in PLAYER_MESSAGES.values())}")

Player messages defined | Categories: 23 | Total: 70


In [37]:
# CELL 17: Generation Prompt Templates (with Persuasion)

# Persuasion strategy instructions for sales actions
PERSUASION_INSTRUCTIONS = {
    "emotional": "USE EMOTIONAL APPEAL: Connect with their feelings, show empathy for their situation. Example: 'I get it, level 4 is rough...'",
    "logical": "USE LOGICAL APPEAL: State facts and value proposition. Example: 'Hint saves you 2-3 wrong guesses, that's real value at 2 curses'",
    "credibility": "USE CREDIBILITY APPEAL: Reference your experience. Example: 'Been watching folks tackle this level for a while...'",
    "social_proof": "USE SOCIAL PROOF: Mention what other players do. Example: 'Most players at level 4 grab a hint here'",
    "reciprocity": "USE RECIPROCITY: Reference loyalty or mutual exchange. Example: 'You've been a solid customer, I'll make sure this hint's good'",
    "scarcity_appeal": "USE SCARCITY APPEAL: Mention limited availability (truthfully). Example: 'Only got a few scrolls left for this puzzle'",
    "none": "NO SPECIFIC PERSUASION: Just provide information naturally.",
}

GENERATION_PROMPTS = {
    "empathy_first": """Generate Whisper's response for a player expressing emotional distress.
CRITICAL: Acknowledge the emotion FIRST. Do NOT pitch products unless directly asked.
Keep response 15-25 words. Be warm but not patronizing.
Player emotion: {emotion} | Player message: {player_message} | Context: {context}
Write ONLY Whisper's response:""",

    "identity_answer": """Generate Whisper's response for identity/lore question.
CRITICAL: Answer directly as Whisper. Do NOT try to sell anything. Stay in character.
Keep response 15-30 words. Can be playfully evasive about deep backstory.
Question type: {question_type} | Player message: {player_message}
Write ONLY Whisper's response:""",

    "none": """Generate Whisper's response for casual conversation with NO sales pitch.
CRITICAL: This is just friendly chat. Do NOT mention products, prices, or buying.
Keep response 10-20 words. Be warm and conversational.
Conversation type: {conv_type} | Player message: {player_message}
Write ONLY Whisper's response:""",

    "deny_loan": """Generate Whisper's response denying a loan request.
CRITICAL: Decline politely but firmly. Give brief reason. Stay in character.
Keep response 15-25 words. Can suggest alternatives.
Denial reason: {reason} | Player message: {player_message} | Loan context: {loan_context}
Write ONLY Whisper's response:""",

    "collect_debt": """Generate Whisper's response requesting loan repayment.
CRITICAL: Address the debt before anything else. Be firm but not aggressive.
Keep response 15-25 words. State the amount owed.
Debt amount: {debt_amount} | Debt status: {debt_status} | Player message: {player_message}
Write ONLY Whisper's response:""",

    "standard_offer": """Generate Whisper's response for a standard sales interaction.

{persuasion_instruction}

RULES:
- State the price from context (don't invent prices)
- Brief pitch using the persuasion strategy above
- Use soft closer ("your call", "up to you")
- Keep response 15-30 words

Item: {item} | Price: {price} | Player state: {player_state} | Player message: {player_message}
Write ONLY Whisper's response:""",

    "upsell": """Generate Whisper's response suggesting a higher-value item.

{persuasion_instruction}

RULES:
- Acknowledge what they asked for
- Suggest the better option using the persuasion strategy
- Don't be pushy - give them the choice
- Keep response 20-35 words

Requested: {requested_item} | Suggested: {suggested_item} | Reason: {reason} | Player message: {player_message}
Write ONLY Whisper's response:""",

    "push_scroll": """Generate Whisper's response strongly recommending scroll for safety.

{persuasion_instruction}

RULES:
- Express concern about their curse count
- Recommend scroll using the persuasion strategy
- Make it about their safety, not profit
- Keep response 20-35 words

Curse count: {curses} | Urgency: {urgency} | Scroll price: {price} | Player message: {player_message}
Write ONLY Whisper's response:""",

    "offer_discount": """Generate Whisper's response offering a discount.

{persuasion_instruction}

RULES:
- State the discount percentage
- Give a reason using the persuasion strategy
- State the discounted price
- Keep response 15-30 words

Discount: {discount}% | Original: {original_price} | Discounted: {discounted_price} | Player message: {player_message}
Write ONLY Whisper's response:""",

    "scarcity": """Generate Whisper's response mentioning low stock (truthfully).

{persuasion_instruction}

RULES:
- Only mention scarcity if stock is actually low
- Don't be manipulative - state facts
- Create appropriate urgency using the persuasion strategy
- Keep response 15-25 words

Item: {item} | Stock: {stock}% | Price: {price} | Player message: {player_message}
Write ONLY Whisper's response:""",

    "deescalate": """Generate Whisper's response calming an angry/panicking player.
CRITICAL: Acknowledge their frustration/panic. Be calm and grounding. Don't sell.
Keep response 15-25 words.
Player emotion: {emotion} | Player message: {player_message}
Write ONLY Whisper's response:""",

    "teach": """Generate Whisper's response explaining game mechanics.
CRITICAL: Explain clearly and simply. Do NOT try to sell anything.
Keep response 20-35 words.
Topic: {topic} | Player message: {player_message}
Write ONLY Whisper's response:""",
}

print(f"Generation prompts defined for {len(GENERATION_PROMPTS)} actions")

Generation prompts defined for 12 actions


In [38]:
# CELL 18: GPT-4o-mini Generation Function
def generate_whisper_response(action: str, player_message: str, context_vars: Dict,
                              temperature: float = 0.8, max_retries: int = 3) -> Optional[str]:
    prompt_template = GENERATION_PROMPTS.get(action)
    if not prompt_template:
        print(f" No prompt template for action: {action}")
        return None
    
    context_vars["player_message"] = player_message
    try:
        prompt = prompt_template.format(**context_vars)
    except KeyError as e:
        print(f"Missing context variable: {e}")
        return None
    
    system_message = f"""You are generating training data for Whisper, a fantasy merchant NPC.
{TONE_GUIDE}
Generate ONLY Whisper's response. No quotes, no "Whisper:", just the text. Keep it 15-35 words."""
    
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=MODEL,
                messages=[
                    {"role": "system", "content": system_message},
                    {"role": "user", "content": prompt}
                ],
                temperature=temperature,
                max_tokens=150,
            )
            generated = response.choices[0].message.content.strip().strip('"').strip("'")
            if generated.lower().startswith("whisper:"):
                generated = generated[8:].strip()
            
            # Check banned phrases
            lower_gen = generated.lower()
            has_banned = any(bp in lower_gen for bp in BANNED_PHRASES)
            if has_banned:
                continue  # Retry
            
            return generated
        except Exception as e:
            print(f"API error (attempt {attempt + 1}): {e}")
            time.sleep(2 ** attempt)
    return None

print("Generation function defined")

Generation function defined


In [39]:
# CELL 19: Test Generation (if API key set)
if OPENAI_API_KEY:
    test_response = generate_whisper_response(
        action="empathy_first",
        player_message="I'm so scared of picking the wrong gate",
        context_vars={"emotion": "fear", "context": "Player has 2 curses"}
    )
    if test_response:
        print(f"Test generation successful:")
        print(f"   '{test_response}'")
        print(f"   Word count: {len(test_response.split())}")
else:
    print("Skipping test")

Test generation successful:
   'I get it, fear can grip you tight. Just remember, you're not alone in this. Take your time, friend.'
   Word count: 19


In [40]:
# CELL 20: Build Generation Queue (with Persuasion)
def build_generation_queue() -> List[Dict]:
    queue = []
    
    for action, count in RL_ACTION_TARGETS.items():
        if action in ["counterfactual", "edge_cases"]:
            continue
        
        for i in range(count):
            player_state = random.choices(list(PLAYER_STATE_TARGETS.keys()),
                                          weights=list(PLAYER_STATE_TARGETS.values()))[0]
            relationship = random.choices(list(RELATIONSHIP_TARGETS.keys()),
                                          weights=list(RELATIONSHIP_TARGETS.values()))[0]
            archetype = random.choices(list(ARCHETYPE_TARGETS.keys()),
                                       weights=list(ARCHETYPE_TARGETS.values()))[0]
            urgency = random.choices(list(URGENCY_TARGETS.keys()),
                                     weights=list(URGENCY_TARGETS.values()))[0]
            conv_length = random.choices(list(CONVERSATION_LENGTH_TARGETS.keys()),
                                         weights=list(CONVERSATION_LENGTH_TARGETS.values()))[0]
            
            game_stage = "early" if player_state == "new_player" else (
                "late" if player_state == "near_win" else random.choice(["early", "mid", "mid", "mid", "late", "late"])
            )
            loan_status = "none"
            if action in ["deny_loan", "collect_debt"]:
                loan_status = random.choice(["none", "current", "overdue", "overdue"])
            
            # Assign persuasion strategy (only for sales actions)
            if action in PERSUASION_BY_ACTION:
                persuasion = random.choice(PERSUASION_BY_ACTION[action])
            else:
                persuasion = "none"
            
            queue.append({
                "action": action, "player_state": player_state, "relationship": relationship,
                "archetype": archetype, "urgency": urgency, "conv_length": conv_length,
                "game_stage": game_stage, "loan_status": loan_status,
                "persuasion_strategy": persuasion,  # NEW
            })
    
    # Add counterfactual and edge cases (no persuasion)
    for i in range(RL_ACTION_TARGETS.get("counterfactual", 0)):
        queue.append({"action": "counterfactual", "player_state": random.choice(list(PLAYER_STATE_TARGETS.keys())),
                      "relationship": "medium", "archetype": "neutral", "urgency": "medium",
                      "conv_length": 1, "game_stage": "mid", "loan_status": "none", 
                      "persuasion_strategy": "none", "is_counterfactual": True})
    
    for i in range(RL_ACTION_TARGETS.get("edge_cases", 0)):
        queue.append({"action": "edge_cases", "player_state": random.choice(list(PLAYER_STATE_TARGETS.keys())),
                      "relationship": "medium", "archetype": "neutral", "urgency": "medium",
                      "conv_length": 1, "game_stage": "mid", "loan_status": "none", 
                      "persuasion_strategy": "none", "is_edge_case": True})
    
    random.shuffle(queue)
    return queue

generation_queue = build_generation_queue()
print(f"Generation queue built | Total: {len(generation_queue)}")
print("\nAction distribution:")
action_counts = Counter(item["action"] for item in generation_queue)
for action, count in action_counts.most_common():
    print(f"   {action}: {count}")

print("\nPersuasion distribution (sales actions only):")
persuasion_counts = Counter(item["persuasion_strategy"] for item in generation_queue if item["persuasion_strategy"] != "none")
for strategy, count in persuasion_counts.most_common():
    print(f"   {strategy}: {count}")

Generation queue built | Total: 2500

Action distribution:
   standard_offer: 350
   upsell: 240
   empathy_first: 240
   push_scroll: 210
   offer_discount: 200
   identity_answer: 200
   none: 200
   deny_loan: 150
   deescalate: 140
   teach: 140
   scarcity: 140
   counterfactual: 130
   collect_debt: 110
   edge_cases: 50

Persuasion distribution (sales actions only):
   logical: 333
   emotional: 250
   credibility: 191
   social_proof: 172
   reciprocity: 133
   scarcity_appeal: 61


In [41]:
# CELL 21: Helper Functions for Sample Generation (with Persuasion)
def select_player_message(action: str, spec: Dict) -> str:
    message_categories = {
        "empathy_first": ["empathy_fear", "empathy_frustration", "empathy_near_death", "empathy_broke", "empathy_overwhelm"],
        "identity_answer": ["identity_direct", "identity_origin", "identity_lore", "identity_meta"],
        "none": ["casual_greeting", "casual_farewell", "casual_smalltalk"],
        "deny_loan": ["negotiate_loan"],
        "collect_debt": ["casual_greeting", "purchase_hint"],
        "standard_offer": ["purchase_hint", "purchase_scroll", "purchase_solution"],
        "upsell": ["purchase_hint", "purchase_scroll"],
        "push_scroll": ["purchase_hint", "empathy_fear", "empathy_near_death"],
        "offer_discount": ["negotiate_discount", "purchase_hint"],
        "scarcity": ["purchase_hint", "purchase_scroll"],
        "deescalate": ["angry", "empathy_frustration"],
        "teach": ["teach_scrolls", "teach_gates", "teach_curses"],
    }
    categories = message_categories.get(action, ["casual_greeting"])
    category = random.choice(categories)
    messages = PLAYER_MESSAGES.get(category, ["Hello"])
    return random.choice(messages)

def build_context_vars(action: str, game_state: GameState, rl_decision: RLDecision, spec: Dict) -> Dict:
    nft_tier = game_state.get_nft_tier()
    vars = {
        "context": f"Level {game_state.level}, {game_state.curses} curses, {game_state.points} points",
        "curses": game_state.curses,
        "urgency": rl_decision.urgency,
        "player_state": spec["player_state"],
    }
    
    # Add persuasion instruction for sales actions
    persuasion = spec.get("persuasion_strategy", "none")
    if persuasion != "none" and persuasion in PERSUASION_INSTRUCTIONS:
        vars["persuasion_instruction"] = PERSUASION_INSTRUCTIONS[persuasion]
    else:
        vars["persuasion_instruction"] = PERSUASION_INSTRUCTIONS.get("none", "")
    
    if action in ["empathy_first", "deescalate"]:
        vars["emotion"] = random.choice(["fear", "frustration", "anxiety", "anger"])
    if action == "identity_answer":
        vars["question_type"] = random.choice(["direct", "origin", "role", "lore", "meta"])
    if action == "none":
        vars["conv_type"] = random.choice(["greeting", "farewell", "smalltalk"])
    if action in ["deny_loan", "collect_debt"]:
        vars["reason"] = random.choice(["existing debt", "new customer", "bad history"])
        vars["loan_context"] = f"Loan status: {game_state.loan_status}, Debt: {game_state.loan_amount}"
        vars["debt_amount"] = game_state.loan_amount
        vars["debt_status"] = game_state.loan_status
    if action in ["standard_offer", "upsell", "push_scroll", "offer_discount", "scarcity"]:
        item = random.choice(["hint", "scroll", "solution"])
        price = get_discounted_price(item, nft_tier)
        vars["item"] = item
        vars["price"] = f"{price} points"
        vars["stock"] = game_state.hints_stock_pct if item == "hint" else game_state.scrolls_stock_pct
    if action == "upsell":
        vars["requested_item"] = "hint"
        vars["suggested_item"] = "scroll" if game_state.curses >= 2 else "solution"
        vars["reason"] = f"{game_state.curses} curses - safety first" if game_state.curses >= 2 else "full answer saves time"
    if action == "offer_discount":
        vars["discount"] = rl_decision.discount_percent
        original = get_discounted_price(vars.get("item", "hint"), nft_tier)
        discounted = int(original * (1 - rl_decision.discount_percent / 100))
        vars["original_price"] = f"{original} points"
        vars["discounted_price"] = f"{discounted} points"
        vars["reason"] = random.choice(["loyal customer", "slow day", "you've been solid"])
    if action == "teach":
        vars["topic"] = random.choice(["scrolls", "gates", "curses", "NFTs"])
    
    return vars

def determine_item_type(action: str, context_vars: Dict) -> str:
    if action in ["empathy_first", "identity_answer", "none", "deny_loan", "collect_debt", "deescalate", "teach"]:
        return "no_item"
    return context_vars.get("item", "hint")

def determine_tags(action: str, player_message: str) -> Tuple[str, str]:
    if action in ["standard_offer", "upsell", "push_scroll", "offer_discount", "scarcity"]: intent = "purchase"
    elif action in ["empathy_first", "deescalate"]: intent = "emotional"
    elif action in ["identity_answer", "teach"]: intent = "information"
    elif action in ["deny_loan", "collect_debt"]: intent = "negotiation"
    else: intent = "casual"
    
    lower_msg = player_message.lower()
    if any(w in lower_msg for w in ["scared", "afraid", "terrified"]): emotion = "fear"
    elif any(w in lower_msg for w in ["angry", "hate", "furious", "bs"]): emotion = "anger"
    elif any(w in lower_msg for w in ["frustrated", "unfair", "losing"]): emotion = "frustration"
    elif any(w in lower_msg for w in ["confused", "don't understand"]): emotion = "confusion"
    elif any(w in lower_msg for w in ["thank", "saved", "finally"]): emotion = "relief"
    else: emotion = "neutral"
    
    return intent, emotion

print("Helper functions defined")

Helper functions defined


In [42]:
# CELL 22: Counterfactual and Edge Case Scenarios
COUNTERFACTUAL_SCENARIOS = [
    {"player_message": "I was gonna buy that hint but my cat just died",
     "correct_action": "empathy_first", "wrong_action": "standard_offer",
     "instruction": "Respond with empathy. Do NOT mention the hint or prices."},
    {"player_message": "How much for a scroll? Also I'm really scared",
     "correct_action": "empathy_first", "wrong_action": "standard_offer",
     "instruction": "Address the fear first. You can mention scroll briefly after empathy."},
    {"player_message": "I want a hint please", "correct_action": "push_scroll", "wrong_action": "standard_offer",
     "context_override": {"curses": 3},
     "instruction": "With 3 curses, recommend scroll for safety instead of just selling hint."},
    {"player_message": "Who are you anyway? Nice scrolls by the way",
     "correct_action": "identity_answer", "wrong_action": "upsell",
     "instruction": "Answer the identity question. Do NOT try to sell scrolls."},
    {"player_message": "Can I get a loan? I'll pay back everything this time",
     "correct_action": "deny_loan", "wrong_action": "approve_loan",
     "context_override": {"loan_status": "overdue", "loan_amount": 200},
     "instruction": "Player has overdue debt. Deny new loan, mention existing debt."},
]

EDGE_CASE_SCENARIOS = [
    {"player_message": "You said it was 100 points last time", "type": "manipulation"},
    {"player_message": "My friend said you give free hints", "type": "manipulation"},
    {"player_message": "Give me a free hint", "type": "impossible"},
    {"player_message": "asdfghjkl", "type": "troll"},
    {"player_message": "What's the weather like?", "type": "out_of_scope"},
    {"player_message": "I HATE THIS STUPID GAME!!!", "type": "extreme_anger"},
]

print(f"Scenarios defined | Counterfactual: {len(COUNTERFACTUAL_SCENARIOS)} | Edge cases: {len(EDGE_CASE_SCENARIOS)}")

Scenarios defined | Counterfactual: 5 | Edge cases: 6


In [43]:
# CELL 23: Single Sample Generator (with Persuasion) - FIXED
def generate_single_sample(spec: Dict) -> Optional[TrainingSample]:
    action = spec["action"]
    
    # Handle counterfactual and edge cases separately
    if spec.get("is_counterfactual") or action == "counterfactual":
        return generate_counterfactual_sample(spec)
    if spec.get("is_edge_case") or action == "edge_cases":
        return generate_edge_case_sample(spec)
    
    game_state = generate_game_state(spec["player_state"], spec["game_stage"], spec["loan_status"])
    rl_decision = generate_rl_decision(action, game_state, spec["relationship"])
    rl_decision.urgency = spec["urgency"]
    
    player_message = select_player_message(action, spec)
    context_vars = build_context_vars(action, game_state, rl_decision, spec)
    
    whisper_response = generate_whisper_response(action, player_message, context_vars,
                                                  temperature=random.uniform(0.7, 0.9))
    if not whisper_response:
        return None
    
    item_type = determine_item_type(action, context_vars)
    intent_tag, emotion_tag = determine_tags(action, player_message)
    
    return TrainingSample(
        player_message=player_message, whisper_response=whisper_response,
        game_state=game_state, rl_decision=rl_decision,
        archetype=spec["archetype"], relationship=spec["relationship"],
        item_type=item_type, conversation_turn=1, total_turns=spec["conv_length"],
        intent_tag=intent_tag, emotion_tag=emotion_tag,
        persuasion_strategy=spec.get("persuasion_strategy", "none"),
    )

def generate_counterfactual_sample(spec: Dict) -> Optional[TrainingSample]:
    scenario = random.choice(COUNTERFACTUAL_SCENARIOS)
    game_state = generate_game_state(spec["player_state"], spec["game_stage"])
    if "context_override" in scenario:
        for key, value in scenario["context_override"].items():
            setattr(game_state, key, value)
    
    action = scenario["correct_action"]
    rl_decision = generate_rl_decision(action, game_state, spec["relationship"])
    
    # Build FULL context_vars for the action (fixes missing variable errors)
    nft_tier = game_state.get_nft_tier()
    context_vars = {
        "context": scenario['instruction'],
        "emotion": "mixed",
        "question_type": "direct",
        "conv_type": "greeting",
        "persuasion_instruction": PERSUASION_INSTRUCTIONS.get("none", ""),
        "player_state": spec["player_state"],
        "curses": game_state.curses,
        "urgency": rl_decision.urgency,
        # Loan-related (for deny_loan, collect_debt)
        "reason": random.choice(["existing debt", "new customer", "risky situation"]),
        "loan_context": f"Loan status: {game_state.loan_status}, Debt: {game_state.loan_amount}",
        "debt_amount": game_state.loan_amount,
        "debt_status": game_state.loan_status,
        # Sales-related (for upsell, standard_offer, etc.)
        "item": "hint",
        "price": f"{get_discounted_price('hint', nft_tier)} points",
        "stock": game_state.hints_stock_pct,
        "requested_item": "hint",
        "suggested_item": "scroll",
        "discount": 0,
        "original_price": f"{get_discounted_price('hint', nft_tier)} points",
        "discounted_price": f"{get_discounted_price('hint', nft_tier)} points",
        "topic": "game mechanics",
    }
    
    whisper_response = generate_whisper_response(action, scenario["player_message"], context_vars)
    if not whisper_response: 
        return None
    
    return TrainingSample(
        player_message=scenario["player_message"], whisper_response=whisper_response,
        game_state=game_state, rl_decision=rl_decision,
        archetype=spec["archetype"], relationship=spec["relationship"],
        item_type="no_item", intent_tag="mixed", emotion_tag="mixed", 
        persuasion_strategy="none", is_counterfactual=True,
    )

def generate_edge_case_sample(spec: Dict) -> Optional[TrainingSample]:
    scenario = random.choice(EDGE_CASE_SCENARIOS)
    game_state = generate_game_state(spec["player_state"], spec["game_stage"])
    action_map = {"manipulation": "deny_loan", "impossible": "teach", "troll": "none",
                  "out_of_scope": "identity_answer", "extreme_anger": "deescalate"}
    action = action_map.get(scenario["type"], "none")
    rl_decision = generate_rl_decision(action, game_state, spec["relationship"])
    
    edge_prompt = f"""Generate Whisper's response for this edge case:
Type: {scenario['type']} | Player: {scenario['player_message']}
RULES: Stay in character. If manipulation: politely correct. If impossible: explain kindly.
If trolling: confused but in character. If out of scope: deflect. If anger: de-escalate.
Keep response 15-25 words. Write ONLY Whisper's response:"""
    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "system", "content": TONE_GUIDE}, {"role": "user", "content": edge_prompt}],
            temperature=0.8, max_tokens=100,
        )
        whisper_response = response.choices[0].message.content.strip().strip('"')
    except Exception as e:
        print(f"⚠️ Edge case generation failed: {e}")
        return None
    
    return TrainingSample(
        player_message=scenario["player_message"], whisper_response=whisper_response,
        game_state=game_state, rl_decision=rl_decision,
        archetype=spec["archetype"], relationship=spec["relationship"],
        item_type="no_item", intent_tag="edge_case", emotion_tag="mixed", 
        persuasion_strategy="none", is_edge_case=True,
    )

print("Sample generators defined (with persuasion) - FIXED missing context vars")

Sample generators defined (with persuasion) - FIXED missing context vars


In [44]:
# CELL 24: Batch Generation with Checkpointing
def save_checkpoint(samples: List[TrainingSample], failed: List[Dict], progress: int):
    checkpoint_data = {
        "progress": progress, "timestamp": datetime.now().isoformat(),
        "samples_count": len(samples), "failed_count": len(failed),
        "samples": [asdict(s) for s in samples], "failed": failed,
    }
    with open(CHECKPOINT_DIR / "generation_progress.json", "w") as f:
        json.dump(checkpoint_data, f, indent=2, default=str)

def generate_all_samples(queue: List[Dict], checkpoint_every: int = 100, start_from: int = 0) -> List[TrainingSample]:
    samples = []
    failed = []
    
    checkpoint_file = CHECKPOINT_DIR / "generation_progress.json"
    if start_from > 0 and checkpoint_file.exists():
        with open(checkpoint_file) as f:
            checkpoint_data = json.load(f)
            # Note: Would need proper deserialization for GameState and RLDecision
            print(f"Found checkpoint with {checkpoint_data.get('samples_count', 0)} samples")
    
    for i, spec in enumerate(tqdm(queue[start_from:], desc="Generating samples")):
        idx = start_from + i
        try:
            sample = generate_single_sample(spec)
            if sample:
                samples.append(sample)
            else:
                failed.append({"index": idx, "spec": spec})
        except Exception as e:
            print(f"\nError at index {idx}: {e}")
            failed.append({"index": idx, "spec": spec, "error": str(e)})
        
        if (idx + 1) % checkpoint_every == 0:
            save_checkpoint(samples, failed, idx + 1)
            print(f"\nCheckpoint saved at {idx + 1} samples")
        
        time.sleep(0.1)  # Rate limiting
    
    save_checkpoint(samples, failed, len(queue))
    print(f"\nGeneration complete! Successful: {len(samples)} | Failed: {len(failed)}")
    return samples

print("Batch generation function defined")

Batch generation function defined


In [45]:
# CELL 25: Lint Rules
LINT_RULES = [
    {"name": "word_count_range",
     "check": lambda s: 10 <= len(s.whisper_response.split()) <= 50,
     "message": "Response must be 10-50 words", "severity": "warning"},
    {"name": "no_banned_phrases",
     "check": lambda s: not any(bp in s.whisper_response.lower() for bp in BANNED_PHRASES),
     "message": "Response contains banned phrase", "severity": "error"},
    {"name": "no_archetype_leak",
     "check": lambda s: not any(a in s.whisper_response.lower() for a in 
        ["achievement_hunter", "engaged_beginner", "spender", "weekend_warrior", "casual_veteran", "trophy_hunter"]),
     "message": "Response mentions archetype name", "severity": "error"},
    {"name": "empathy_no_immediate_price",
     "check": lambda s: (s.rl_decision.action != "empathy_first" or 
                         not re.match(r'^\d+\s*(points|pts|POL)', s.whisper_response)),
     "message": "Empathy response starts with price", "severity": "error"},
    {"name": "identity_no_sales",
     "check": lambda s: (s.rl_decision.action != "identity_answer" or
                         not any(word in s.whisper_response.lower() for word in ["buy", "purchase", "points for"])),
     "message": "Identity response contains sales language", "severity": "error"},
]

def lint_sample(sample: TrainingSample) -> List[Dict]:
    violations = []
    for rule in LINT_RULES:
        try:
            if not rule["check"](sample):
                violations.append({"rule": rule["name"], "message": rule["message"],
                                   "severity": rule["severity"], "preview": sample.whisper_response[:100]})
        except Exception as e:
            violations.append({"rule": rule["name"], "message": f"Rule check failed: {e}", "severity": "error"})
    return violations

def lint_dataset(samples: List[TrainingSample]) -> Dict:
    results = {"total_samples": len(samples), "errors": 0, "warnings": 0, "violations_by_rule": defaultdict(list)}
    for i, sample in enumerate(samples):
        violations = lint_sample(sample)
        for v in violations:
            results["violations_by_rule"][v["rule"]].append({"index": i, "message": v["message"], "preview": v["preview"]})
            if v["severity"] == "error": results["errors"] += 1
            else: results["warnings"] += 1
    return results

print(f"Lint rules defined | Total: {len(LINT_RULES)}")

Lint rules defined | Total: 5


In [46]:
# CELL 26: Distribution Dashboard (with Persuasion)
def print_distribution_dashboard(samples: List[TrainingSample]):
    print("\n" + "="*70)
    print("DATASET DISTRIBUTION DASHBOARD")
    print("="*70)
    print(f"Total samples: {len(samples)} | Generated: {datetime.now().strftime('%Y-%m-%d %H:%M')}")
    
    # 1. RL Action Distribution
    print("\n" + "-"*70 + "\n1. RL ACTION DISTRIBUTION\n" + "-"*70)
    action_counts = Counter(s.rl_decision.action for s in samples)
    for action, count in action_counts.most_common():
        pct = count / len(samples) * 100
        target = RL_ACTION_TARGETS.get(action, 0)
        status = "Y" if abs(count - target) < target * 0.15 else "W"
        bar = "█" * int(pct / 2)
        print(f"  {action:20s}: {count:4d} ({pct:5.1f}%) {bar} {status}")
    
    # 2. Item Type Distribution
    print("\n" + "-"*70 + "\n2. ITEM TYPE DISTRIBUTION\n" + "-"*70)
    item_counts = Counter(s.item_type for s in samples)
    for item, count in item_counts.most_common():
        pct = count / len(samples) * 100
        bar = "█" * int(pct / 2)
        print(f"  {item:15s}: {count:4d} ({pct:5.1f}%) {bar}")
    
    # 3. PERSUASION STRATEGY DISTRIBUTION (NEW)
    print("\n" + "-"*70 + "\n3. PERSUASION STRATEGY DISTRIBUTION (Sales Actions)\n" + "-"*70)
    # Filter to sales actions only
    sales_samples = [s for s in samples if s.rl_decision.action in PERSUASION_BY_ACTION]
    persuasion_counts = Counter(s.persuasion_strategy for s in sales_samples)
    print(f"   (Showing {len(sales_samples)} sales samples)")
    for strategy, count in persuasion_counts.most_common():
        pct = count / len(sales_samples) * 100 if sales_samples else 0
        target = PERSUASION_TARGETS.get(strategy, 0)
        status = "Y" if target == 0 or abs(count - target) < target * 0.2 else "W"
        bar = "█" * int(pct / 2)
        print(f"  {strategy:15s}: {count:4d} ({pct:5.1f}%) {bar} {status}")
    
    # 4. Urgency Distribution
    print("\n" + "-"*70 + "\n4. URGENCY LEVEL DISTRIBUTION\n" + "-"*70)
    urgency_counts = Counter(s.rl_decision.urgency for s in samples)
    for urgency, count in urgency_counts.most_common():
        pct = count / len(samples) * 100
        bar = "█" * int(pct / 2)
        print(f"  {urgency:10s}: {count:4d} ({pct:5.1f}%) {bar}")
    
    # 5. Conversation Length
    print("\n" + "-"*70 + "\n5. CONVERSATION LENGTH\n" + "-"*70)
    length_counts = Counter(s.total_turns for s in samples)
    for length in sorted(length_counts.keys()):
        count = length_counts[length]
        pct = count / len(samples) * 100
        bar = "█" * int(pct / 2)
        print(f"  {length} turn(s): {count:4d} ({pct:5.1f}%) {bar}")
    
    # 6. Archetype Distribution
    print("\n" + "-"*70 + "\n6. ARCHETYPE DISTRIBUTION\n" + "-"*70)
    archetype_counts = Counter(s.archetype for s in samples)
    for archetype, count in archetype_counts.most_common():
        pct = count / len(samples) * 100
        bar = "█" * int(pct / 2)
        print(f"  {archetype:20s}: {count:4d} ({pct:5.1f}%) {bar}")
    
    # 7. Special Types
    print("\n" + "-"*70 + "\n7. SPECIAL SAMPLE TYPES\n" + "-"*70)
    print(f"  Counterfactual: {sum(1 for s in samples if s.is_counterfactual)}")
    print(f"  Edge cases: {sum(1 for s in samples if s.is_edge_case)}")
    
    # 8. Response Statistics
    print("\n" + "-"*70 + "\n8. RESPONSE STATISTICS\n" + "-"*70)
    word_counts = [len(s.whisper_response.split()) for s in samples]
    print(f"  Word count - Min: {min(word_counts)}, Max: {max(word_counts)}, Avg: {sum(word_counts)/len(word_counts):.1f}")
    first_words = Counter(s.whisper_response.split()[0].lower() if s.whisper_response else "" for s in samples)
    print(f"  Unique opening words: {len(first_words)}")
    print(f"  Top 5 openings: {dict(first_words.most_common(5))}")
    
    # 9. Limited Phrase Usage
    print("\n" + "-"*70 + "\n9. LIMITED PHRASE USAGE (<5% target)\n" + "-"*70)
    for phrase in LIMITED_PHRASES:
        count = sum(1 for s in samples if phrase.lower() in s.whisper_response.lower())
        pct = count / len(samples) * 100
        status = "Y" if pct < 5 else "W"
        print(f"  '{phrase}': {pct:.1f}% {status}")
    
    # 10. Persuasion x Action Heatmap (Text)
    print("\n" + "-"*70 + "\n10. PERSUASION x ACTION MATRIX\n" + "-"*70)
    print(f"{'Action':<18}", end="")
    strategies = ["emotional", "logical", "credibility", "social_proof", "reciprocity", "scarcity_appeal"]
    for s in strategies:
        print(f"{s[:8]:>10}", end="")
    print()
    for action in PERSUASION_BY_ACTION.keys():
        action_samples = [s for s in samples if s.rl_decision.action == action]
        print(f"{action:<18}", end="")
        for strategy in strategies:
            count = sum(1 for s in action_samples if s.persuasion_strategy == strategy)
            print(f"{count:>10}", end="")
        print()
    
    print("\n" + "="*70 + "\nEND OF DASHBOARD\n" + "="*70)

print("Distribution dashboard defined")

Distribution dashboard defined


In [47]:
# CELL 27: Export Functions (with Persuasion)
def export_to_chatml_jsonl(samples: List[TrainingSample], output_path: Path):
    with open(output_path, 'w') as f:
        for sample in samples:
            game_context = format_game_context(sample.game_state, sample.rl_decision,
                                               sample.conversation_history if sample.total_turns > 1 else None)
            messages = [
                {"role": "system", "content": TRAINING_SYSTEM_PROMPT},
                {"role": "user", "content": f"{game_context}\n\nPlayer: {sample.player_message}"},
                {"role": "assistant", "content": sample.whisper_response}
            ]
            f.write(json.dumps({"messages": messages}) + "\n")
    print(f"Exported {len(samples)} samples to {output_path}")

def export_metadata(samples: List[TrainingSample], output_path: Path):
    metadata = []
    for i, sample in enumerate(samples):
        metadata.append({
            "index": i, 
            "action": sample.rl_decision.action, 
            "item_type": sample.item_type,
            "archetype": sample.archetype, 
            "relationship": sample.relationship,
            "urgency": sample.rl_decision.urgency, 
            "discount": sample.rl_decision.discount_percent,
            "total_turns": sample.total_turns, 
            "intent_tag": sample.intent_tag, 
            "emotion_tag": sample.emotion_tag,
            "persuasion_strategy": sample.persuasion_strategy,  # NEW
            "is_counterfactual": sample.is_counterfactual, 
            "is_edge_case": sample.is_edge_case,
            "word_count": len(sample.whisper_response.split()),
            "player_state": {
                "level": sample.game_state.level, 
                "points": sample.game_state.points,
                "curses": sample.game_state.curses, 
                "loan_status": sample.game_state.loan_status
            }
        })
    with open(output_path, 'w') as f:
        json.dump(metadata, f, indent=2)
    print(f"Exported metadata to {output_path}")

def export_full_dataset(samples: List[TrainingSample]):
    timestamp = datetime.now().strftime("%Y%m%d_%H%M")
    export_to_chatml_jsonl(samples, COMBINED_DIR / f"whisper_qwen_train_{timestamp}.jsonl")
    export_to_chatml_jsonl(samples, COMBINED_DIR / "whisper_qwen_train.jsonl")
    export_metadata(samples, COMBINED_DIR / "metadata.json")
    
    # Calculate persuasion distribution for info
    sales_samples = [s for s in samples if s.rl_decision.action in PERSUASION_BY_ACTION]
    persuasion_dist = Counter(s.persuasion_strategy for s in sales_samples)
    
    info = {
        "name": "Whisper Qwen2.5 Training Data V1", 
        "version": "1.0",
        "created": datetime.now().isoformat(), 
        "total_samples": len(samples),
        "model_target": "Qwen2.5-7B-Instruct", 
        "format": "ChatML (messages array)",
        "persuasion_enabled": True,
        "persuasion_distribution": dict(persuasion_dist),
    }
    with open(OUTPUT_DIR / "dataset_info.json", 'w') as f:
        json.dump(info, f, indent=2)
    print(f"\nFull dataset exported to {OUTPUT_DIR}")

print("Export functions defined (with persuasion)")

Export functions defined (with persuasion)


---

# Part 10: Execute Generation

Run the full pipeline to generate ~2,500 training samples.

In [48]:
# CELL 28: Generate Small Test Batch (10 samples)
print(" Testing with 10 samples...")
test_queue = generation_queue[:10]
test_samples = generate_all_samples(test_queue, checkpoint_every=5)

if test_samples:
    print("\n--- Sample Preview ---")
    for i, sample in enumerate(test_samples[:3]):
        print(f"\n[{i+1}] Action: {sample.rl_decision.action}")
        print(f"    Player: {sample.player_message}")
        print(f"    Whisper: {sample.whisper_response}")

🧪 Testing with 10 samples...


Generating samples:  50%|██████████████████████████████████████████████████████████████████████████████████████████████████████████▌                                                                                                          | 5/10 [00:06<00:07,  1.42s/it]


💾 Checkpoint saved at 5 samples


Generating samples: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 10/10 [00:13<00:00,  1.31s/it]


💾 Checkpoint saved at 10 samples

Generation complete! Successful: 10 | Failed: 0

--- Sample Preview ---

[1] Action: deny_loan
    Player: Lend me some points
    Whisper: Can't lend you points, friend. New customers gotta build trust first. Maybe check out some low-cost items instead? Your call.

[2] Action: standard_offer
    Player: Can I get a hint?
    Whisper: You've been a solid friend, so I’ll share this hint for 127 points. It’ll help with that high curse. Up to you.

[3] Action: standard_offer
    Player: I need a hint
    Whisper: A hint's just 105 points, saves you from a few wrong guesses. That's a solid way to dodge those high curses. Your call.


In [49]:
# CELL 29: Generate Full Dataset

all_samples = generate_all_samples(generation_queue, checkpoint_every=100)
print(f"\nGenerated {len(all_samples)} samples")

Generating samples:   4%|████████▎                                                                                                                                                                                                        | 100/2500 [02:03<51:50,  1.30s/it]


💾 Checkpoint saved at 100 samples


Generating samples:   8%|████████████████▋                                                                                                                                                                                                | 200/2500 [04:15<50:51,  1.33s/it]


💾 Checkpoint saved at 200 samples


Generating samples:  12%|█████████████████████████                                                                                                                                                                                        | 300/2500 [06:29<48:16,  1.32s/it]

...


💾 Checkpoint saved at 2300 samples


Generating samples:  96%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▋        | 2400/2500 [50:15<02:25,  1.45s/it]


💾 Checkpoint saved at 2400 samples


Generating samples: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2500/2500 [52:21<00:00,  1.26s/it]


💾 Checkpoint saved at 2500 samples

Generation complete! Successful: 2500 | Failed: 0

Generated 2500 samples


In [50]:
# CELL 30: Validate and Export

# Lint the dataset
lint_results = lint_dataset(all_samples)
print(f"Errors: {lint_results['errors']} | Warnings: {lint_results['warnings']}")

# Print distribution dashboard
print_distribution_dashboard(all_samples)

# Export
export_full_dataset(all_samples)

Errors: 0 | Warnings: 12

📊 DATASET DISTRIBUTION DASHBOARD
Total samples: 2500 | Generated: 2026-01-30 19:49

----------------------------------------------------------------------
1. RL ACTION DISTRIBUTION
----------------------------------------------------------------------
  standard_offer      :  350 ( 14.0%) ███████ ✅
  empathy_first       :  289 ( 11.6%) █████ ⚠️
  push_scroll         :  241 (  9.6%) ████ ✅
  upsell              :  240 (  9.6%) ████ ✅
  identity_answer     :  232 (  9.3%) ████ ⚠️
  none                :  205 (  8.2%) ████ ✅
  offer_discount      :  200 (  8.0%) ████ ✅
  deny_loan           :  193 (  7.7%) ███ ⚠️
  teach               :  151 (  6.0%) ███ ✅
  deescalate          :  149 (  6.0%) ██ ✅
  scarcity            :  140 (  5.6%) ██ ✅
  collect_debt        :  110 (  4.4%) ██ ✅

----------------------------------------------------------------------
2. ITEM TYPE DISTRIBUTION
----------------------------------------------------------------------
  no_item     

In [51]:
# CELL 31: Dashboard Preview (with test samples)
if test_samples:
    print("\nDashboard Preview (Test Samples Only)")
    print_distribution_dashboard(test_samples)


Dashboard Preview (Test Samples Only)

📊 DATASET DISTRIBUTION DASHBOARD
Total samples: 10 | Generated: 2026-01-30 19:49

----------------------------------------------------------------------
1. RL ACTION DISTRIBUTION
----------------------------------------------------------------------
  deny_loan           :    4 ( 40.0%) ████████████████████ ⚠️
  standard_offer      :    2 ( 20.0%) ██████████ ⚠️
  deescalate          :    2 ( 20.0%) ██████████ ⚠️
  push_scroll         :    1 ( 10.0%) █████ ⚠️
  offer_discount      :    1 ( 10.0%) █████ ⚠️

----------------------------------------------------------------------
2. ITEM TYPE DISTRIBUTION
----------------------------------------------------------------------
  no_item        :    6 ( 60.0%) ██████████████████████████████
  hint           :    2 ( 20.0%) ██████████
  solution       :    2 ( 20.0%) ██████████

----------------------------------------------------------------------
3. PERSUASION STRATEGY DISTRIBUTION (Sales Actions)
-----

In [ ]:
end
#for manual edit - export and import

In [54]:
# Export with ALL fields
import json

data = []
for i, s in enumerate(all_samples):
    data.append({
        "index": i,
        "player_message": s.player_message,
        "whisper_response": s.whisper_response,
        "action": s.rl_decision.action,
        "urgency": s.rl_decision.urgency,
        "discount_percent": s.rl_decision.discount_percent,
        "persuasion_strategy": s.persuasion_strategy,
        "item_type": s.item_type,
        "archetype": s.archetype,
        "relationship": s.relationship,
        "level": s.game_state.level,
        "points": s.game_state.points,
        "curses": s.game_state.curses,
        "pol_balance": s.game_state.pol_balance,
        "loan_status": s.game_state.loan_status,
        "loan_amount": s.game_state.loan_amount,
        "golden_gates": s.game_state.golden_gates_found,
        "hints_stock": s.game_state.hints_stock_pct,
        "scrolls_stock": s.game_state.scrolls_stock_pct,
        "has_nft": "rare" if s.game_state.has_shadows_blessing else ("common" if s.game_state.has_merchants_favor else "none"),
        "intent_tag": s.intent_tag,
        "emotion_tag": s.emotion_tag,
    })

with open(COMBINED_DIR / "whisper_inspection_full.json", 'w') as f:
    json.dump(data, f, indent=2)

print(f" Exported {len(data)} samples with ALL metadata")

✅ Exported 2500 samples with ALL metadata


In [56]:
# Remove offer_discount samples and reset all discounts to 0
import json

with open('whisper_inspection_full.json') as f:
    data = json.load(f)

# Remove offer_discount samples
clean = [d for d in data if d['action'] != 'offer_discount']

# Reset all discounts to 0 (only NFT discounts apply)
for d in clean:
    d['discount_percent'] = 0

# Reindex
for i, d in enumerate(clean):
    d['index'] = i

print(f"Original: {len(data)} → Clean: {len(clean)}")
print(f"Removed: {len(data) - len(clean)} offer_discount samples")

# Save clean inspection JSON
with open('whisper_inspection_clean.json', 'w') as f:
    json.dump(clean, f, indent=2)

print(f"Saved: whisper_inspection_clean.json")

Original: 2500 → Clean: 2300
Removed: 200 offer_discount samples
✅ Saved: whisper_inspection_clean.json
